# Diamond Creek ETF Arbitrage — Sample Portfolio

**40 pairs** | Long underlying @ ~$650K | Short leveraged ETF @ ~$325K | Whole shares

Generated for Clear Street prime brokerage onboarding.

In [ ]:
import pandas as pd
import yfinance as yf
from datetime import datetime

LONG_TARGET = 650_000
SHORT_TARGET = 325_000

pairs = [
    ("SOXL", "SOXX"),   #  1  $11,939M
    ("TSLL", "TSLA"),   #  2   $5,010M
    ("GDXU", "GDX"),    #  3   $4,089M
    ("NVDL", "NVDA"),   #  4   $3,934M
    ("NUGT", "GDX"),    #  5   $1,709M
    ("KORU", "EWY"),    #  6   $1,155M
    ("GGLL", "GOOGL"),  #  7   $1,097M
    ("MUU",  "MU"),     #  8   $1,084M
    ("JNUG", "GDXJ"),   #  9     $910M
    ("BITX", "IBIT"),   # 10     $905M
    ("YINN", "FXI"),    # 11     $775M
    ("ETHU", "ETHA"),   # 12     $707M
    ("MSFU", "MSFT"),   # 13     $628M
    ("AMDL", "AMD"),    # 14     $566M
    ("NVDU", "NVDA"),   # 15     $540M
    ("PLTU", "PLTR"),   # 16     $490M
    ("NVDX", "NVDA"),   # 17     $474M
    ("CONL", "COIN"),   # 18     $470M
    ("PTIR", "PLTR"),   # 19     $429M
    ("METU", "META"),   # 20     $392M
    ("AMZU", "AMZN"),   # 21     $385M
    ("BITU", "IBIT"),   # 22     $351M
    ("ERX",  "XLE"),    # 23     $315M
    ("GUSH", "XOP"),    # 24     $296M
    ("TSLT", "TSLA"),   # 25     $267M
    ("MULL", "MU"),     # 26     $239M
    ("CWEB", "KWEB"),   # 27     $220M
    ("BMNU", "BMNR"),   # 28     $220M
    ("FBL",  "META"),   # 29     $216M
    ("LITX", "LITE"),   # 30     $211M
    ("ASTX", "ASTS"),   # 31     $210M
    ("ETHT", "ETHA"),   # 32     $179M
    ("AVL",  "AVGO"),   # 33     $174M
    ("APPX", "APP"),    # 34     $170M
    ("TSLR", "TSLA"),   # 35     $168M
    ("UNHG", "UNH"),    # 36     $158M
    ("AAPU", "AAPL"),   # 37     $155M
    ("SOLT", "SOEZ"),   # 38     $139M
    ("CHAU", "ASHR"),   # 39     $131M
    ("INTW", "INTC"),   # 40     $121M
]

all_tickers = sorted(set(t for etf, und in pairs for t in [etf, und]))
print(f"Fetching prices for {len(all_tickers)} tickers...")
data = yf.download(all_tickers, period="1d", progress=False)
prices = data["Close"].iloc[-1].to_dict() if "Close" in data.columns.get_level_values(0) else {}
if not prices:
    prices = {t: float(data["Close"].iloc[-1]) for t in all_tickers if t in data.columns}

missing = [t for t in all_tickers if t not in prices or pd.isna(prices.get(t))]
if missing:
    print(f"Retrying {len(missing)} missing: {missing}")
    for t in missing:
        try:
            h = yf.Ticker(t).history(period="5d")
            if not h.empty:
                prices[t] = float(h["Close"].iloc[-1])
        except:
            pass

print(f"Prices as of {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Got prices for {sum(1 for t in all_tickers if t in prices and pd.notna(prices.get(t)))}/{len(all_tickers)} tickers")

In [ ]:
rows = []
for etf, und in pairs:
    p_und = prices.get(und)
    p_etf = prices.get(etf)
    if p_und and pd.notna(p_und) and p_und > 0:
        long_shares = round(LONG_TARGET / p_und)
        long_amt = long_shares * p_und
    else:
        long_shares = long_amt = p_und = 0

    if p_etf and pd.notna(p_etf) and p_etf > 0:
        short_shares = round(SHORT_TARGET / p_etf)
        short_amt = short_shares * p_etf
    else:
        short_shares = short_amt = p_etf = 0

    rows.append({
        "Ticker": und, "Side": "Long", "Shares": int(long_shares),
        "Price": round(p_und, 2), "Amount": round(long_amt, 2),
        "Pair": f"{und}/{etf}",
    })
    rows.append({
        "Ticker": etf, "Side": "Short", "Shares": int(short_shares),
        "Price": round(p_etf, 2), "Amount": round(short_amt, 2),
        "Pair": f"{und}/{etf}",
    })

portfolio = pd.DataFrame(rows)

longs = portfolio[portfolio["Side"] == "Long"]
shorts = portfolio[portfolio["Side"] == "Short"]
total_long = longs["Amount"].sum()
total_short = shorts["Amount"].sum()
gross = total_long + total_short
net = total_long - total_short

print(f"{'='*70}")
print(f"  PORTFOLIO SUMMARY")
print(f"{'='*70}")
print(f"  Pairs:        {len(pairs)}")
print(f"  Positions:    {len(portfolio)}")
print(f"  Total Long:   ${total_long:>14,.2f}")
print(f"  Total Short:  ${total_short:>14,.2f}")
print(f"  Gross:        ${gross:>14,.2f}")
print(f"  Net:          ${net:>14,.2f}")
print(f"  Net / Gross:  {net/gross:.1%}" if gross > 0 else "")
print(f"{'='*70}")

display_df = portfolio[["Ticker", "Side", "Shares", "Price", "Amount"]].copy()
display_df["Amount"] = display_df["Amount"].map("${:,.2f}".format)
display_df["Price"] = display_df["Price"].map("${:,.2f}".format)
display_df

In [ ]:
from pathlib import Path

out = Path("data/backtest")
out.mkdir(parents=True, exist_ok=True)

export = portfolio[["Ticker", "Side", "Shares", "Price", "Amount", "Pair"]].copy()
export.to_csv(out / "clear_street_sample_portfolio.csv", index=False)
export.to_excel(out / "clear_street_sample_portfolio.xlsx", index=False)
print(f"Exported to {out}/clear_street_sample_portfolio.csv/.xlsx")